# Test Urban Environment

## Import Libraries

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch
from torchrl.envs import EnvBase
from torchrl.data import (
    Composite, 
    Unbounded, 
    Bounded,
    Stacked
    
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import numpy as np

# from urbanmarl.envs.urbanmarl_env import UrbanEnv
from urbanmarl.envs.base_env import UrbanEnv

## Create Environment

In [ ]:
from torchrl.envs.utils import (
    _classproperty,
    _selective_unsqueeze,
    check_marl_grouping,
    MarlGroupMapType,
)
from tensordict import LazyStackedTensorDict, TensorDict, TensorDictBase
from tensordict.base import _default_is_leaf, _is_leaf_nontensor

In [ ]:
config = {
    "num_uavs": 3,
    "num_ues": 20,
    "area_size": (500, 500),
    "max_time_slots": 50,
    "max_horizontal_speed": 49.0,
    "max_vertical_speed": 12.0,
    "max_transmit_power": 5.0,
    "frequency_ghz": 29.0,
    "g2a_bandwidth": 10e6,
    "noise_figure_db": 7.0,
    "agents": ["agent_0", "agent_1", "agent_2"]
    # "agents": ["uav_0", "uav_1", 'ue_0']
}
num_envs = 2
scenario = "navigate"
# scenario = "uav_ue_los"
env = UrbanEnv(
    num_envs = num_envs, # batch_size=torch.Size([2])
    continuous_actions = True,
    seed=0,
    device=device,
    scenario=scenario,
    **config)

In [ ]:
tensordict = env.reset()
t = torch.zeros((env.batch_size[0], 1), dtype=torch.bool, device=env.device)
t[0,0] = True
source = {"_reset": t}
r = TensorDict(
    source=source,
    batch_size=env.batch_size,
    device=env.device,
)
env.reset(tensordict=r)

In [ ]:
torch.norm(env.uav_velocity, dim=-1)

In [ ]:
env.uav_collisions.float() * 2

In [ ]:
batch_idx =0
num_pos=env.n_uavs
min_z = 20.0
max_z = 150.0
xy_indices = (env._env.height_maps[batch_idx]==0).nonzero()
shuffled_positions = torch.randperm(len(xy_indices))
selected_positions = shuffled_positions[:num_pos]
sampled_indices = xy_indices[selected_positions]
sampled_indices[:, 0] = sampled_indices[:, 0].float() + env._env.x_min
sampled_indices[:, 1] = sampled_indices[:, 1].float() + env._env.y_min
z_column = min_z + torch.rand(size=(num_pos, 1), device=env._env.device) * (max_z - min_z)
# torch.cat([sampled_indices.view(num_pos, 2),  
#                   z_column.view(num_pos, 1)], 
#                  device=env._env.device, dim=1)
torch.cat([sampled_indices.view(num_pos, 2),  
                  z_column.view(num_pos, 1)], 
                 dim=-1)

## Test specs

In [ ]:
env.check_env_specs()

## Test Rollout

In [ ]:
n_rollout_steps = 3
rollout = env.rollout(n_rollout_steps)

print("Shape of the rollout TensorDict:", rollout.batch_size)
print(f"rollout: shape: {rollout.shape}, len: {len(rollout.shape)}")
for key in rollout.keys(True, True):
    print(f"{key}, shape: {rollout[key].shape}")
    

In [ ]:
assert rollout["done"].shape == torch.Size((env.batch_size[0], n_rollout_steps, 1)), f"{rollout["done"].shape} not equal {torch.Size((env.batch_size[0], n_rollout_steps, 1))}"


In [ ]:
env.uav_ue_los.float()

In [ ]:
los = env.uav_ue_los.float().mean(dim=1).mean(dim=-1, keepdim=True)
los

In [ ]:
1/los

In [ ]:
torch.norm(env.uav_velocity, dim=-1, keepdim=True)

In [ ]:
env.uav_collisions.float().mean(dim=1)

In [ ]:
env.uav_collisions.squeeze(-1)

In [ ]:
{
    "collisions_per_env": env.uav_collisions.sum(dim=-1, keepdim=True).shape,
    "velocity": torch.norm(env.uav_velocity, dim=-1, keepdim=True).shape
}

In [ ]:
env.state_spec.keys(True, True)
env.full_observation_spec.keys(True, True)

In [ ]:
print('agent names:', env.agent_names)
print('agent group map:', env.group_map)

## Test spec

In [ ]:
env.check_env_specs()

In [ ]:
from tensordict.base import _default_is_leaf, _is_leaf_nontensor

In [ ]:
fake_tensordict = env.fake_tensordict()

print(f"fake_tensordict: shape: {fake_tensordict.shape}, len: {len(fake_tensordict.shape)}")
for key in fake_tensordict.keys(True, True):
    print(f"{key}, shape: {fake_tensordict[key].shape}")
    if key == ('agents', 'action', 'h_distance'):
        print(fake_tensordict[key][..., 0])

In [ ]:
specs = env.specs
for key in specs.keys(True, True):
    print(f"{key}, shape: {specs[key].shape}")

In [ ]:
tensordict= env.reset()
print(f"tensordict: shape: {tensordict.shape}, len: {len(tensordict.shape)}")
for key in tensordict.keys(True, True):
    print(f"{key}, shape: {tensordict[key].shape}")
print("----")

fake_tensordict = env.fake_tensordict()
fake_tensordict
print(f"fake_tensordict: shape: {fake_tensordict.shape}, len: {len(fake_tensordict.shape)}")
for key in fake_tensordict.keys(True, True):
    print(f"{key}, shape: {fake_tensordict[key].shape}")
print("----")
# state_spec keys may optionally appear in "next" if _step returns them,
# so exclude them from all comparisons to avoid false positives in either
# direction (real has them but fake doesn't, or vice versa).
state_spec_next_keys = {
    ("next", *k) if isinstance(k, tuple) else ("next", k)
    for k in env.state_spec.keys(True, True)
}
print("state_spec_next_keys")
print(state_spec_next_keys)
print("----")
# eliminate empty containers, excluding state_spec keys from "next"
fake_tensordict_select = fake_tensordict.select(
    *[
        k
        for k in fake_tensordict.keys(True, True, is_leaf=_default_is_leaf)
        if k not in state_spec_next_keys
    ]
)
print(f"fake_tensordict_select: shape: {fake_tensordict_select.shape}, len: {len(fake_tensordict_select.shape)}")
for key in fake_tensordict_select.keys(True, True):
    print(f"{key}, shape: {fake_tensordict_select[key].shape}")
print("----")
return_contiguous = True
break_when_any_done = 'both'
real_tensordict = env.rollout(
    1,
    return_contiguous=return_contiguous,
    tensordict=tensordict,
    auto_reset=tensordict is None,
    break_when_any_done=break_when_any_done,
)
print(f"real_tensordict: shape: {real_tensordict.shape}, len: {len(real_tensordict.shape)}")
for key in real_tensordict.keys(True, True):
    print(f"{key}, shape: {real_tensordict[key].shape}")
print("----")

real_tensordict_select = real_tensordict.select(
    *[
        k
        for k in real_tensordict.keys(True, True, is_leaf=_default_is_leaf)
        if k not in state_spec_next_keys
    ]
)
print(f"real_tensordict_select: shape: {real_tensordict_select.shape}, len: {len(real_tensordict_select.shape)}")
for key in real_tensordict_select.keys(True, True):
    print(f"{key}, shape: {real_tensordict_select[key].shape}")
print("----")

(
    torch.zeros_like(fake_tensordict_select)
    != torch.zeros_like(real_tensordict_select)
).any()

In [ ]:
tensordict= env.reset()
shape = torch.broadcast_shapes(fake_tensordict.shape, tensordict.shape)
shape

In [ ]:
env._has_dynamic_specs

In [ ]:
env.check_env_specs()

In [ ]:
env.specs

## Reset

In [ ]:
tensordict = env.reset()
tensordict.keys(True, True)

In [ ]:
print('shape:', tensordict.shape)
for group in env.group_map:
    print('position shape:', tensordict[(group, 'observation')].shape)
# print('capability shape:', tensordict[('agents', 'observation', 'capability')].shape)
# print('load shape:', tensordict[('agents', 'observation', 'load')].shape)
# print('battery shape:', tensordict[('agents', 'observation', 'battery')].shape)
# print('energy shape:', tensordict[('agents', 'observation', 'energy')].shape)
# print('state shape:', tensordict[('state')].shape)
# print('done shape:', tensordict[('done')].shape)

In [ ]:
tensordict.get(('agents', 'observation'))

## Test step

In [ ]:
action = env.action_spec.sample()
print(action.keys(True, True))
print(action.get(('agents', 'action')))

In [ ]:
action = env.action_spec.sample()
new_td = tensordict.update(action)
next_td = env.step(new_td)

In [ ]:
print(f"next_td: {next_td.shape}, len: {len(next_td.shape)}")
for key in next_td.keys(True, True):
    print(f"{key} shape: {next_td[key].shape}")

In [ ]:
next_td.keys(True, True)

In [ ]:
next_td.keys(True, True, sort=True)
for group in env.group_map.keys():
    print(next_td.get(("next", group)).keys())
    assert "reward" in next_td.get(("next", group)).keys()

In [ ]:
for key in next_td.keys(True, True):
    print(f"{key}, size: {next_td[key].shape}")

## Rollout

For fun, let’s see what a simple random rollout looks like. You can call env.rollout(n_steps) and get an overview of what the environment inputs and outputs look like. Actions will automatically be drawn at random from the action spec domain.


In [ ]:
n_rollout_steps = 5
rollout = env.rollout(n_rollout_steps)
# print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)
print(f"rollout: shape: {rollout.shape}, len: {len(rollout.shape)}")
for key in rollout.keys(True, True):
    print(f"{key}, shape: {rollout[key].shape}")

In [ ]:
UrbanEnv.__mro__

In [ ]:
print("action_spec:", env.full_action_spec)
print("reward_spec:", env.full_reward_spec)
print("done_spec:", env.full_done_spec)
print("observation_spec:", env.observation_spec)

In [ ]:
# print("group.map:", env.group_map)
# print("observation_spec:", env.observation_spec)
# print("state_spec:", env.state_spec)
# print("action_spec:", env.action_spec)
# print("reward_spec:", env.reward_spec)
# print("", )
# print("", )

In [ ]:
env.reset()

## Plot Urban Map Example

In [ ]:
env._env.plot(batch_idx = 1)